# Exploration des Survey Property Maps — DP2 (USDF)

Ce notebook accède aux **survey property maps** de Rubin LSST DP2 dans le butler `dp2_prep`.

Deux types de produits sont disponibles dans la collection `LSSTCam/runs/DRP/v30_0_4/DM-54249` :

1. **`SurveyWidePropertyMapPlot`** — images PNG/PDF des cartes générées par le pipeline (section 3–5).
2. **`consolidated_map`** — objets `HealSparseMap` interrogeables (section 6–10, si disponibles).

**À adapter** : noms de collection et de dataset type selon ce qui est effectivement présent.

- **Author:** Sylvie Dagoret-Campagne  
- **Creation date:** 2026-03-11  
- **Environment:** USDF RSP — kernel `LSST` (`lsst_distrib` stack)  
- **Reference DP1:** `203_1_Survey_property_maps.ipynb`  
- **Reference DP2 access:** `2026-03-07_AccessToDP2.ipynb`

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
from pathlib import Path
from IPython.display import Image, display

# LSST middleware
from lsst.daf.butler import Butler

try:
    import skyproj
    HAS_SKYPROJ = True
    print("skyproj disponible")
except ImportError:
    HAS_SKYPROJ = False
    print("skyproj non disponible — visualisations all-sky désactivées")

try:
    import healsparse
    HAS_HEALSPARSE = True
    print("healsparse disponible")
except ImportError:
    HAS_HEALSPARSE = False
    print("healsparse non disponible")

## 2. Configuration

In [ ]:
REPO             = 'dp2_prep'
COLLECTION       = 'LSSTCam/runs/DRP/v30_0_4/DM-54249'
SKYMAP_NAME      = 'lsst_cells_v2'
INSTRUMENT       = 'LSSTCam'
BAND_REF         = 'r'
RA_CEN           = 53.13    # ECDFS
DEC_CEN          = -28.10

# Liste des PropertyMapPlot trouvés dans la collection
PLOT_DATASET_TYPES = [
    'propertyMapSurvey_deepCoadd_dcr_ddec_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_dcr_dra_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_dcr_e1_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_dcr_e2_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_epoch_consolidated_map_max_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_epoch_consolidated_map_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_epoch_consolidated_map_min_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_exposure_time_consolidated_map_sum_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_e1_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_e2_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_maglim_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_size_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_sky_background_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_sky_noise_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
]

butler   = Butler(REPO)
registry = butler.registry
print(f"Butler instancié : {REPO}")

## 3. Inspecter les dimensions d'un PropertyMapPlot

Avant de récupérer un plot, il faut connaître ses dimensions butler  
(typiquement `band` et/ou `skymap`, parfois aucune).

In [ ]:
print("Dimensions des PropertyMapPlot disponibles :")
print("-" * 80)
for name in PLOT_DATASET_TYPES:
    try:
        dt = butler.get_dataset_type(name)
        dims = dt.dimensions
        print(f"{name}")
        print(f"    dimensions         : {dims}")
        print(f"    dimensions requises: {dims.required}")
        print()
    except Exception as e:
        print(f"  ERREUR {name}: {e}")

## 4. Lister les références (DatasetRef) disponibles pour un plot

Cette cellule liste toutes les instances disponibles d'un dataset type Plot  
dans la collection, ce qui permet de connaître les valeurs de dimensions  
(bands, skymaps, ...) pour lesquelles des plots ont été produits.

In [ ]:
# Choisir un plot à explorer
PLOT_NAME = 'propertyMapSurvey_deepCoadd_psf_maglim_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot'

refs = list(registry.queryDatasets(PLOT_NAME, collections=COLLECTION))
print(f"Nombre de références pour '{PLOT_NAME}' : {len(refs)}")
print()
for ref in refs:
    print(f"  ref id      : {ref.id}")
    print(f"  dataId      : {dict(ref.dataId)}")
    print(f"  run         : {ref.run}")
    print()

In [ ]:
# Récupérer l'URI (chemin physique) du fichier plot
for ref in refs:
    uri = butler.getURI(ref)
    print(f"  dataId : {dict(ref.dataId)}")
    print(f"  URI    : {uri}")
    print()

## 5. Affichage d'un PropertyMapPlot

Les `SurveyWidePropertyMapPlot` sont des fichiers image (PNG ou PDF).  
Trois méthodes d'accès sont proposées ci-dessous selon la situation.

### 5.1 Méthode A — `butler.get()` direct (si le StorageClass le supporte)

In [ ]:
# Tenter butler.get() — retourne un objet dépendant du StorageClass
# (ButlerImage, bytes, ou objet matplotlib selon la configuration du pipeline)
if refs:
    ref0 = refs[0]
    print(f"Tentative butler.get() sur : {dict(ref0.dataId)}")
    try:
        plot_obj = butler.get(ref0)
        print(f"  type retourné : {type(plot_obj)}")
        print(plot_obj)
    except Exception as e:
        print(f"  butler.get() échoué : {type(e).__name__}: {e}")
        print("  → utiliser la méthode B (URI) à la place")

### 5.2 Méthode B — lecture via URI (chemin fichier direct)

In [ ]:
# Récupérer l'URI et lire l'image directement
if refs:
    ref0 = refs[0]
    uri  = butler.getURI(ref0)
    path = uri.path   # chemin POSIX sur le système de fichiers
    print(f"Fichier : {path}")
    print(f"Suffix  : {Path(path).suffix}")

    suffix = Path(path).suffix.lower()

    if suffix in ('.png', '.jpg', '.jpeg'):
        # Affichage via matplotlib
        img = mpimg.imread(path)
        fig, ax = plt.subplots(figsize=(14, 8))
        ax.imshow(img)
        ax.axis('off')
        dataId_str = ', '.join(f"{k}={v}" for k, v in dict(ref0.dataId).items())
        ax.set_title(f"{PLOT_NAME}\n{dataId_str}", fontsize=10)
        plt.tight_layout()
        plt.show()

    elif suffix == '.pdf':
        # Affichage inline via IPython
        display(Image(filename=path))

    else:
        print(f"Extension inconnue '{suffix}' — URI brute : {uri}")

### 5.3 Méthode C — accès par dataId explicite (band, skymap)

In [ ]:
# Si les dimensions incluent 'band' et/ou 'skymap', on peut accéder directement
# en passant les paramètres de dataId à butler.get() ou butler.getURI()

# Exemple : récupérer le plot en bande r
try:
    uri_r = butler.getURI(
        PLOT_NAME,
        dataId={'band': BAND_REF, 'skymap': SKYMAP_NAME},
        collections=COLLECTION
    )
    print(f"URI bande {BAND_REF} : {uri_r}")
    path_r = uri_r.path
    suffix = Path(path_r).suffix.lower()
    if suffix in ('.png', '.jpg', '.jpeg'):
        img = mpimg.imread(path_r)
        fig, ax = plt.subplots(figsize=(14, 8))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f"{PLOT_NAME} — band={BAND_REF}", fontsize=10)
        plt.tight_layout()
        plt.show()
except Exception as e:
    print(f"Accès par dataId explicite échoué : {type(e).__name__}: {e}")
    print("→ vérifier les dimensions avec la section 3, ou utiliser la méthode B")

## 6. Boucle sur toutes les bandes pour un plot donné

In [ ]:
BANDS = ['u', 'g', 'r', 'i', 'z', 'y']

refs_all = list(registry.queryDatasets(PLOT_NAME, collections=COLLECTION))
print(f"Plots disponibles pour '{PLOT_NAME}' :")
for ref in refs_all:
    print(f"  dataId={dict(ref.dataId)}")

In [ ]:
# Afficher tous les plots disponibles en grille (1 colonne par bande)
images_found = []
for ref in refs_all:
    uri  = butler.getURI(ref)
    path = uri.path
    suffix = Path(path).suffix.lower()
    if suffix in ('.png', '.jpg', '.jpeg'):
        images_found.append((dict(ref.dataId), path))

if images_found:
    n = len(images_found)
    fig, axes = plt.subplots(1, n, figsize=(14 * n, 8))
    if n == 1:
        axes = [axes]
    for ax, (did, path) in zip(axes, images_found):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.axis('off')
        label = ', '.join(f"{k}={v}" for k, v in did.items())
        ax.set_title(label, fontsize=9)
    plt.suptitle(PLOT_NAME.replace('propertyMapSurvey_', ''), fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Aucune image PNG trouvée — vérifier les URIs avec la section 4")

## 7. Galerie de tous les PropertyMapPlot

Parcourir tous les dataset types de type Plot et afficher un plot par type  
(première référence disponible).

In [ ]:
# Collecter les URIs de tous les plots (une instance par type)
gallery = []  # list of (short_name, dataId_dict, path)

for name in PLOT_DATASET_TYPES:
    refs = list(registry.queryDatasets(name, collections=COLLECTION))
    if not refs:
        print(f"  ✗ {name} — aucune référence")
        continue
    # Prendre de préférence la bande r si disponible
    ref_r = next((r for r in refs if dict(r.dataId).get('band') == BAND_REF), refs[0])
    try:
        uri  = butler.getURI(ref_r)
        path = uri.path
        short = (
            name
            .replace('propertyMapSurvey_deepCoadd_', '')
            .replace('_SurveyWidePropertyMapPlot', '')
        )
        gallery.append((short, dict(ref_r.dataId), path))
        print(f"  ✓ {short}  [{dict(ref_r.dataId)}]")
    except Exception as e:
        print(f"  ✗ {name} — URI introuvable : {e}")

print(f"\n{len(gallery)} plots récupérés")

In [ ]:
# Afficher la galerie complète
png_gallery = [
    (short, did, path)
    for short, did, path in gallery
    if Path(path).suffix.lower() in ('.png', '.jpg', '.jpeg')
]

if png_gallery:
    ncols = 2
    nrows = int(np.ceil(len(png_gallery) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(20, 8 * nrows))
    axes = np.array(axes).flatten()

    for idx, (short, did, path) in enumerate(png_gallery):
        img = mpimg.imread(path)
        axes[idx].imshow(img)
        axes[idx].axis('off')
        band_label = did.get('band', '?')
        axes[idx].set_title(f"{short}\nband={band_label}", fontsize=8)

    for idx in range(len(png_gallery), len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle(f"Survey Property Map Plots — DP2 — collection : {COLLECTION}",
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("Aucune image PNG dans la galerie — les fichiers sont peut-être des PDF.")
    print("Utiliser display(Image(filename=path)) pour les PDF.")
    for short, did, path in gallery:
        print(f"  {short}: {path}")

## 8. Recherche des HealSparseMap (consolidated_map) correspondantes

En plus des plots, on cherche si les objets `HealSparseMap` bruts  
(suffixe `consolidated_map` sans `Plot`) sont aussi disponibles  
— ce qui permettrait d'interroger les valeurs par position.

In [ ]:
# Dériver les noms consolidated_map depuis les noms Plot
# En supprimant le préfixe 'propertyMapSurvey_' et le suffixe '_SurveyWidePropertyMapPlot'
raw_map_names = [
    name
    .replace('propertyMapSurvey_', '')
    .replace('_SurveyWidePropertyMapPlot', '')
    for name in PLOT_DATASET_TYPES
]

print("Recherche des HealSparseMap brutes (consolidated_map) :")
available_hspmaps = []
for name in raw_map_names:
    try:
        dt = butler.get_dataset_type(name)
        has_data = registry.queryDatasets(
            name, collections=COLLECTION
        ).any(execute=False, exact=False)
        status = "✓" if has_data else "enregistré mais vide"
        if has_data:
            available_hspmaps.append(name)
        print(f"  {status}  {name}")
        print(f"      dimensions: {dt.dimensions.required}")
    except Exception as e:
        print(f"  ✗  {name}  — {type(e).__name__}: {e}")

print(f"\n→ {len(available_hspmaps)} HealSparseMap disponibles")

In [ ]:
# Récupérer une HealSparseMap si disponible
hspmap = None
if available_hspmaps:
    MAP_NAME = available_hspmaps[0]   # à remplacer par le nom souhaité
    print(f"Récupération de : {MAP_NAME}  (band={BAND_REF})")
    try:
        hspmap = butler.get(MAP_NAME, band=BAND_REF, collections=COLLECTION)
        print(f"  type : {type(hspmap)}")
        print(hspmap)
    except Exception as e:
        print(f"  Erreur : {type(e).__name__}: {e}")
else:
    print("Aucune HealSparseMap brute disponible dans cette collection.")
    print("Seuls les plots sont accessibles — voir sections 3–7.")

## 9. Interrogation d'une HealSparseMap (si disponible)

Section active uniquement si une `HealSparseMap` a pu être récupérée à la section 8.

In [ ]:
if hspmap is not None:
    # Valeur au centre ECDFS
    val_center = hspmap.get_values_pos(RA_CEN, DEC_CEN)
    sentinel   = hspmap.get_values_pos(180.0, +60.0)
    print(f"Carte : {MAP_NAME}  ({BAND_REF}-band)")
    print(f"  Valeur ECDFS (RA={RA_CEN}, Dec={DEC_CEN}) : {val_center}")
    print(f"  Valeur sentinelle (hors couverture)         : {sentinel}")
else:
    print("HealSparseMap non disponible — section ignorée.")

In [ ]:
if hspmap is not None:
    span_dec = 0.75
    span_ra  = span_dec / np.cos(np.deg2rad(DEC_CEN))
    ra  = np.linspace(RA_CEN  - span_ra,  RA_CEN  + span_ra,  250)
    dec = np.linspace(DEC_CEN - span_dec, DEC_CEN + span_dec, 250)
    x, y = np.meshgrid(ra, dec)
    values = hspmap.get_values_pos(x, y).astype(float)
    values[values == float(sentinel)] = np.nan

    fig, ax = plt.subplots(figsize=(7, 5))
    pcm = ax.pcolormesh(x, y, values, cmap='viridis')
    fig.colorbar(pcm, ax=ax)
    ax.set_xlabel("RA (deg)")
    ax.set_ylabel("Dec (deg)")
    ax.set_title(f"{MAP_NAME}\n{BAND_REF}-band — ECDFS", fontsize=10)
    ax.invert_xaxis()
    plt.tight_layout()
    plt.show()

    if HAS_SKYPROJ:
        fig, ax = plt.subplots(figsize=(10, 5))
        sp = skyproj.McBrydeSkyproj(ax=ax, lon_0=RA_CEN)
        sp.draw_hspmap(hspmap)
        sp.draw_colorbar(label=f"{MAP_NAME} ({BAND_REF}-band)")
        ax.set_title(f"All-sky — {MAP_NAME} ({BAND_REF}-band)", fontsize=11)
        plt.tight_layout()
        plt.show()

## 10. Dump complet des dataset types de la collection (hors bookkeeping)

In [ ]:
EXCLUDE = ('_config', '_log', '_metadata', '_resource_usage', 'Metric', 'metric')

science_dtypes = []
for dt in registry.queryDatasetTypes():
    if any(s in dt.name for s in EXCLUDE):
        continue
    try:
        if registry.queryDatasets(dt, collections=COLLECTION).any(execute=False, exact=False):
            science_dtypes.append(dt.name)
    except Exception:
        pass

science_dtypes.sort()
print(f"Dataset types disponibles dans '{COLLECTION}' ({len(science_dtypes)}) :")
for name in science_dtypes:
    print(" ", name)